# Phase 3: Data Validation Framework

**Project:** Automated Data Quality & ETL Framework  
**Dataset:** Online Retail.xlsx

This notebook performs schema, completeness, uniqueness, validity, date,
business-rule, and outlier checks before the cleaning stage.

Two reports are generated:

1. `validation_summary.csv` — concise GitHub-ready validation overview.
2. `validation_detailed_report.csv` — complete list of validation checks.
3. `data_types_reference.csv` — observed source-column data types.


## Validation Objectives

This phase verifies that the raw dataset is suitable for downstream processing.

The checks cover:

- Dataset dimensions and expected schema
- Missing values in important fields
- Exact duplicates and repeated business line items
- Invalid quantities and unit prices
- Cancelled invoices
- Invalid or future invoice dates
- Missing or blank stock codes
- Text-formatting inconsistencies
- Quantity and price outliers
- Negative and zero revenue

Records flagged as `Review` are not automatically assumed to be errors.
Some may represent legitimate returns, cancellations, adjustments, or unusual
business activity that must be handled during the cleaning phase.


## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Define Project Paths

Absolute paths are used because Jupyter may run from a different working directory.

In [2]:
# Main project folder
PROJECT_ROOT = Path(
    r"C:\Users\chatu\OneDrive\Desktop\Automated-Data-Quality-ETL-Framework"
)

# Raw dataset path
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Online Retail.xlsx"

# Validation report output folder
VALIDATION_REPORT_PATH = PROJECT_ROOT / "reports" / "validation"
VALIDATION_REPORT_PATH.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset was not found at: {RAW_DATA_PATH}"
    )

print("Input file confirmed: data/raw/Online Retail.xlsx")
print("Validation reports will be saved in: reports/validation/")


Input file confirmed: data/raw/Online Retail.xlsx
Validation reports will be saved in: reports/validation/


## 3. Load the Dataset

In [3]:
# Load the raw Excel dataset
df = pd.read_excel(RAW_DATA_PATH)

# Display first five rows
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 4. Basic Dataset Checks

In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 541909
Columns: 8


In [5]:
# Review column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## 5. Create Validation Report Container

All validation results are stored in one list and converted into a clean report at the end.

In [6]:
validation_results = []


def add_validation_result(
    check_name,
    issue_count,
    status=None,
    category="General",
    details=""
):
    """Add one standardized result to the detailed validation report."""
    issue_count = int(issue_count)

    if status is None:
        status = "Pass" if issue_count == 0 else "Review"

    validation_results.append({
        "Category": category,
        "Validation_Check": check_name,
        "Issue_Count": issue_count,
        "Status": status,
        "Details": details,
    })


## 6. Missing Value Validation

In [7]:
# Count missing values in each column
missing_values = df.isnull().sum()

missing_values

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [8]:
# Store total missing values across all columns
add_validation_result(
    check_name="Total Missing Values",
    issue_count=missing_values.sum(),
    category="Completeness",
    details="Total null values found across the dataset."
)

# Store missing values by column
for column, count in missing_values.items():
    add_validation_result(
        check_name=f"Missing Values - {column}",
        issue_count=count,
        category="Completeness",
        details=f"Null values found in {column}."
    )


## 7. Duplicate Row Validation

In [9]:
# Complete duplicate rows across all columns
duplicate_rows = int(df.duplicated().sum())

add_validation_result(
    check_name="Duplicate Rows",
    issue_count=duplicate_rows,
    category="Uniqueness",
    details="Fully duplicated records across all source columns."
)

duplicate_rows


5268

## 8. Duplicate Business Line Item Validation

An invoice can contain multiple products, so duplicate `InvoiceNo` alone is not an error. This check looks for repeated line items using invoice, stock code, quantity, and price.

In [10]:
duplicate_line_items = int(
    df.duplicated(
        subset=["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]
    ).sum()
)

add_validation_result(
    check_name="Duplicate Line Items",
    issue_count=duplicate_line_items,
    category="Uniqueness",
    details=(
        "Repeated business line items using InvoiceNo, StockCode, "
        "Quantity, and UnitPrice."
    )
)

duplicate_line_items


5271

## 9. Quantity Validation

Negative or zero quantity records usually represent cancellations, returns, or invalid sales records. They are flagged for review before cleaning.

In [11]:
invalid_quantity = df[df["Quantity"] <= 0]

cancelled_invoices = df[
    df["InvoiceNo"]
    .astype(str)
    .str.startswith("C", na=False)
]

add_validation_result(
    check_name="Invalid Quantity",
    issue_count=len(invalid_quantity),
    status="Review",
    category="Business Rule",
    details=(
        "Rows where Quantity is less than or equal to zero. "
        "These may represent returns or cancellations."
    )
)

add_validation_result(
    check_name="Cancelled Invoices",
    issue_count=len(cancelled_invoices),
    status="Review",
    category="Business Rule",
    details="Invoice numbers beginning with C, indicating cancellations."
)

print("Invalid Quantity:", len(invalid_quantity))
print("Cancelled Invoices:", len(cancelled_invoices))


Invalid Quantity: 10624
Cancelled Invoices: 9288


## 10. Unit Price Validation

In [12]:
negative_price = df[df["UnitPrice"] < 0]
zero_price = df[df["UnitPrice"] == 0]

add_validation_result(
    check_name="Negative Unit Price",
    issue_count=len(negative_price),
    category="Validity",
    details="Rows where UnitPrice is below zero."
)

add_validation_result(
    check_name="Zero Unit Price",
    issue_count=len(zero_price),
    status="Review",
    category="Business Rule",
    details=(
        "Rows where UnitPrice is zero. These may be free items, "
        "adjustments, or data issues."
    )
)

print("Negative Unit Price Count:", len(negative_price))
print("Zero Unit Price Count:", len(zero_price))


Negative Unit Price Count: 2
Zero Unit Price Count: 2515


## 11. Customer, Description, and Country Validation

In [13]:
missing_customer = int(df["CustomerID"].isnull().sum())
missing_description = int(df["Description"].isnull().sum())
missing_country = int(df["Country"].isnull().sum())

add_validation_result(
    "Missing CustomerID",
    missing_customer,
    "Review",
    "Completeness",
    "Rows where CustomerID is missing."
)
add_validation_result(
    "Missing Description",
    missing_description,
    None,
    "Completeness",
    "Rows where product Description is missing."
)
add_validation_result(
    "Missing Country",
    missing_country,
    None,
    "Completeness",
    "Rows where Country is missing."
)

print("Missing CustomerID:", missing_customer)
print("Missing Description:", missing_description)
print("Missing Country:", missing_country)


Missing CustomerID: 135080
Missing Description: 1454
Missing Country: 0


## 12. Invoice Date Validation

In [14]:
# Convert InvoiceDate to datetime. Invalid parsing becomes NaT.
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

invalid_invoice_dates = int(df["InvoiceDate"].isnull().sum())
future_dates = df[df["InvoiceDate"] > pd.Timestamp.today()]

add_validation_result(
    "Invalid InvoiceDate",
    invalid_invoice_dates,
    None,
    "Validity",
    "Rows where InvoiceDate could not be converted to a valid datetime."
)
add_validation_result(
    "Future InvoiceDate",
    len(future_dates),
    None,
    "Validity",
    "Rows where InvoiceDate is greater than today's date."
)

print("Invalid InvoiceDate:", invalid_invoice_dates)
print("Future InvoiceDate:", len(future_dates))


Invalid InvoiceDate: 0
Future InvoiceDate: 0


## 13. Stock Code Validation

In [15]:
invalid_stock_code = df[
    df["StockCode"].isnull()
    | (df["StockCode"].astype(str).str.strip() == "")
]

add_validation_result(
    check_name="Invalid StockCode",
    issue_count=len(invalid_stock_code),
    category="Validity",
    details="Rows where StockCode is missing or blank."
)

invalid_stock_code.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


## 14. Text Space Validation

This identifies descriptions with leading or trailing spaces.

In [16]:
description_space_count = int(
    (
        df["Description"].fillna("")
        != df["Description"].fillna("").str.strip()
    ).sum()
)

add_validation_result(
    check_name="Description Leading or Trailing Spaces",
    issue_count=description_space_count,
    category="Consistency",
    details="Rows where Description has leading or trailing spaces."
)

description_space_count


113453

## 15. Data Type Documentation

This section records observed data types. It is documentation, not a pass/fail check.

In [17]:
data_types_df = pd.DataFrame({
    "Column": df.dtypes.index,
    "Data_Type": df.dtypes.astype(str).values
})

data_types_df

,Column,Data_Type
0,InvoiceNo,object
1,StockCode,object
2,Description,object
3,Quantity,int64
4,InvoiceDate,datetime64[ns]
5,UnitPrice,float64
6,CustomerID,float64
7,Country,object


## 16. Outlier Validation Using IQR

The IQR method flags unusually high or low values. These are not automatically wrong, but they should be reviewed before transformation.

In [18]:
def get_iqr_outliers(dataframe, column):
    """Return outlier rows for a numeric column using the IQR method."""
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = dataframe[
        (dataframe[column] < lower_bound)
        | (dataframe[column] > upper_bound)
    ]

    return outliers, lower_bound, upper_bound

In [19]:
quantity_outliers, quantity_lower, quantity_upper = get_iqr_outliers(
    df, "Quantity"
)

add_validation_result(
    check_name="Quantity Outliers",
    issue_count=len(quantity_outliers),
    status="Review",
    category="Outlier",
    details=(
        f"Quantity values outside the IQR range: "
        f"{quantity_lower:.2f} to {quantity_upper:.2f}."
    )
)

print("Quantity Outliers:", len(quantity_outliers))
print("Lower Bound:", quantity_lower)
print("Upper Bound:", quantity_upper)


Quantity Outliers: 58619
Lower Bound: -12.5
Upper Bound: 23.5


In [20]:
price_outliers, price_lower, price_upper = get_iqr_outliers(
    df, "UnitPrice"
)

add_validation_result(
    check_name="UnitPrice Outliers",
    issue_count=len(price_outliers),
    status="Review",
    category="Outlier",
    details=(
        f"UnitPrice values outside the IQR range: "
        f"{price_lower:.2f} to {price_upper:.2f}."
    )
)

print("UnitPrice Outliers:", len(price_outliers))
print("Lower Bound:", price_lower)
print("Upper Bound:", price_upper)


UnitPrice Outliers: 39627
Lower Bound: -3.0700000000000003
Upper Bound: 8.45


## 17. Revenue Validation

Revenue is calculated as `Revenue = Quantity * UnitPrice`.

Negative revenue usually appears because quantity is negative. These rows require review before cleaning.

In [21]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

negative_revenue = df[df["Revenue"] < 0]
zero_revenue = df[df["Revenue"] == 0]

add_validation_result(
    check_name="Negative Revenue",
    issue_count=len(negative_revenue),
    status="Review",
    category="Business Rule",
    details=(
        "Rows where calculated Revenue is less than zero. "
        "Usually caused by negative Quantity."
    )
)

add_validation_result(
    check_name="Zero Revenue",
    issue_count=len(zero_revenue),
    status="Review",
    category="Business Rule",
    details=(
        "Rows where calculated Revenue is zero. Usually caused by "
        "zero UnitPrice or zero Quantity."
    )
)

print("Negative Revenue:", len(negative_revenue))
print("Zero Revenue:", len(zero_revenue))


Negative Revenue: 9290
Zero Revenue: 2515


## 18. Create Final Validation Reports

The notebook creates two validation outputs:

- A concise summary for documentation and GitHub screenshots.
- A complete detailed report for auditability and troubleshooting.


In [22]:
# Complete detailed validation report
validation_detailed_df = pd.DataFrame(validation_results)

validation_detailed_df = validation_detailed_df.sort_values(
    by=["Status", "Category", "Issue_Count"],
    ascending=[True, True, False]
).reset_index(drop=True)

# Concise GitHub-ready validation summary
summary_rows = [
    ("Dataset Overview", "Total Rows", len(df), "Info"),
    ("Dataset Overview", "Total Columns", df.shape[1], "Info"),
    ("Completeness", "Missing CustomerID", missing_customer,
     "Review" if missing_customer else "Pass"),
    ("Completeness", "Missing Description", missing_description,
     "Review" if missing_description else "Pass"),
    ("Completeness", "Missing Country", missing_country,
     "Review" if missing_country else "Pass"),
    ("Uniqueness", "Duplicate Rows", duplicate_rows,
     "Review" if duplicate_rows else "Pass"),
    ("Uniqueness", "Duplicate Line Items", duplicate_line_items,
     "Review" if duplicate_line_items else "Pass"),
    ("Business Rule", "Invalid Quantity", len(invalid_quantity),
     "Review" if len(invalid_quantity) else "Pass"),
    ("Business Rule", "Cancelled Invoices", len(cancelled_invoices),
     "Review" if len(cancelled_invoices) else "Pass"),
    ("Validity", "Negative Unit Price", len(negative_price),
     "Review" if len(negative_price) else "Pass"),
    ("Business Rule", "Zero Unit Price", len(zero_price),
     "Review" if len(zero_price) else "Pass"),
    ("Validity", "Invalid InvoiceDate", invalid_invoice_dates,
     "Review" if invalid_invoice_dates else "Pass"),
    ("Validity", "Future InvoiceDate", len(future_dates),
     "Review" if len(future_dates) else "Pass"),
    ("Business Rule", "Negative Revenue", len(negative_revenue),
     "Review" if len(negative_revenue) else "Pass"),
    ("Business Rule", "Zero Revenue", len(zero_revenue),
     "Review" if len(zero_revenue) else "Pass"),
]

validation_summary_df = pd.DataFrame(
    summary_rows,
    columns=["Category", "Validation_Check", "Issue_Count", "Status"]
)

validation_summary_df


,Category,Validation_Check,Issue_Count,Status
0,Dataset Overview,Total Rows,541909,Info
1,Dataset Overview,Total Columns,9,Info
2,Completeness,Missing CustomerID,135080,Review
3,Completeness,Missing Description,1454,Review
4,Completeness,Missing Country,0,Pass
5,Uniqueness,Duplicate Rows,5268,Review
6,Uniqueness,Duplicate Line Items,5271,Review
7,Business Rule,Invalid Quantity,10624,Review
8,Business Rule,Cancelled Invoices,9288,Review
9,Validity,Negative Unit Price,2,Review


## 19. Save Validation Outputs

The concise summary, detailed validation report, and data type reference are
saved as separate CSV files in `reports/validation/`.


In [24]:
validation_summary_file = (
    VALIDATION_REPORT_PATH / "validation_summary.csv"
)
validation_detailed_file = (
    VALIDATION_REPORT_PATH / "validation_detailed_report.csv"
)
data_types_file = (
    VALIDATION_REPORT_PATH / "data_types_reference.csv"
)

validation_summary_df.to_csv(validation_summary_file, index=False)
validation_detailed_df.to_csv(validation_detailed_file, index=False)
data_types_df.to_csv(data_types_file, index=False)

print("=" * 65)
print("VALIDATION REPORTS GENERATED SUCCESSFULLY")
print("=" * 65)
print("Summary: reports/validation/validation_summary.csv")
print("Detailed: reports/validation/validation_detailed_report.csv")
print("Data types: reports/validation/data_types_reference.csv")


VALIDATION REPORTS GENERATED SUCCESSFULLY
Summary: reports/validation/validation_summary.csv
Detailed: reports/validation/validation_detailed_report.csv
Data types: reports/validation/data_types_reference.csv


## 20. Phase 3 Conclusion

Phase 3 is complete when:

- The notebook runs from top to bottom without errors.
- `validation_summary.csv` contains the concise validation overview.
- `validation_detailed_report.csv` contains every validation check.
- `data_types_reference.csv` documents the observed source data types.
- Records marked `Review` are addressed through the cleaning rules in Phase 4.
